
# Phase 3C.02 — Predictive Analysis

## Objectif

Évaluer le pouvoir prédictif du signal de Time-Series Momentum à 12 mois sur les échantillons Train et Validation.

L'analyse utilisera des métriques de classification directionnelle, notamment Accuracy, Balanced Accuracy, Recall et confusion matrices, ainsi que des baselines.

Avant l'évaluation, la logique de nettoyage développée en Phase 3C.01 sera transformée en fonction réutilisable.

**Périmètre :** Train (2010–2017) et Validation (2018–2021). Le Test (2022–2025) reste exclu.


## 1. Environnement et préparation des données

### Objectif

Reconstruire les données nécessaires à l'analyse prédictive du Time-Series Momentum à partir des fonctions réutilisables développées en Phase 3C.01.

Cette étape comprend :

- La configuration de l'environnement et les imports.
- La reconstruction de l'univers des 16 ETF.
- Le téléchargement et la préparation des prix mensuels.
- Le calcul du momentum à 12 mois.
- La construction des signaux directionnels et des targets à un mois.
- Le découpage temporel Train / Validation.
- Le nettoyage des observations à l'aide de `clean_signals_and_targets()`.

**Périmètre :** Train (2010–2017) et Validation (2018–2021). Le Test (2022–2025) reste strictement exclu.

L'objectif est d'obtenir des datasets propres, alignés et reproductibles avant l'évaluation des performances prédictives.



### 1.1. Configuration de l'environnement

Configuration du chemin d'accès au projet et importation des fonctions réutilisables depuis les modules `src.data` et `src.signals`.

Le rechargement des modules permet d'utiliser leurs dernières modifications sans redémarrer le kernel.


In [242]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.signals

importlib.reload(src.data)
importlib.reload(src.signals)

from src.data import download_prices, to_monthly_prices
from src.signals import calculate_momentum, generate_signals, clean_signals_and_targets


### 1.2. Univers d'investissement et acquisition des données

Reconstitution de l'univers de recherche composé de 16 ETF répartis entre quatre classes d'actifs : Equities, Commodities, Bonds et Forex.

Les prix ajustés sont téléchargés sur la période 2008–2021. L'historique de 2008–2009 permet l'initialisation des indicateurs, tandis que les données à partir de 2022 restent exclues pour préserver l'échantillon Test.


In [243]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

print(tickers)
print(len(tickers))

prices = download_prices(tickers, "2008-01-01", "2022-01-01", auto_adjust=True, progress=False)
print(prices.shape)
print(prices.index.min())
print(prices.index.max())

['SPY', 'VGK', 'EWJ', 'EEM', 'USO', 'DBA', 'DBB', 'DBC', 'IEF', 'TLT', 'LQD', 'HYG', 'FXE', 'FXY', 'FXB', 'FXA']
16
(3526, 16)
2008-01-02 00:00:00
2021-12-31 00:00:00



### 1.3. Préparation et validation des prix mensuels

Conversion des prix quotidiens en observations mensuelles à partir du dernier prix disponible de chaque mois.

Vérification des dimensions, de la couverture temporelle et de l'absence de valeurs manquantes avant la construction des signaux.


In [244]:
monthly_prices = to_monthly_prices(prices)
print(monthly_prices.shape)
print(monthly_prices.index.min())
print(monthly_prices.index.max())
print(monthly_prices.isna().sum().sum())

(168, 16)
2008-01-31 00:00:00
2021-12-31 00:00:00
0



### 1.4. Construction du momentum et des signaux directionnels

Calcul du momentum historique sur 12 mois, conformément à l'horizon principal défini dans le protocole de recherche.

Le momentum est ensuite transformé en signal directionnel :
- `+1` : tendance passée positive.
- `-1` : tendance passée négative.
- `0` : momentum nul.

Les 12 premières observations mensuelles ne disposent pas d'un historique suffisant pour calculer le momentum et sont donc manquantes.


In [245]:

momentum_12m = calculate_momentum(monthly_prices, lookback=12)

print(momentum_12m.head(15))
print(momentum_12m.shape)
print(momentum_12m.isna().sum().sum())

signals_12m = generate_signals(momentum_12m)
signals_12m

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29       NaN       NaN       NaN       NaN       NaN       NaN   
2008-03-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-04-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-05-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-06-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-07-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-08-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-09-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-10-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-11-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-12-31       NaN       NaN       N

Ticker,DBA,DBB,DBC,EEM,EWJ,FXA,FXB,FXE,FXY,HYG,IEF,LQD,SPY,TLT,USO,VGK
Date,,,,,,,,,,,,,,,,
2008-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-02-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-31,1.0,1.0,1.0,1.0,1.0,-1.0,1.0,-1.0,-1.0,1.0,-1.0,1.0,1.0,-1.0,1.0,1.0
2021-09-30,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,1.0,1.0,-1.0,1.0,1.0
2021-10-31,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,1.0,1.0,-1.0,1.0,1.0



### 1.5. Construction des forward returns et des targets

Les forward returns représentent les rendements réalisés au cours du mois suivant la formation du signal.

Les targets correspondent à la direction de ces rendements :
- `+1` : rendement futur positif.
- `-1` : rendement futur négatif.
- `0` : rendement futur nul.

L'alignement temporel est essentiel : un signal calculé à la fin du mois `t` doit être comparé au rendement réalisé entre `t` et `t+1`, afin d'éviter tout look-ahead bias.


In [246]:
forward_returns_1m = monthly_prices.pct_change().shift(-1)
print(forward_returns_1m.head(3))
print(forward_returns_1m.tail(3))

targets_1m = generate_signals(forward_returns_1m)
print(targets_1m.head(3))
print(targets_1m.tail(3))

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31  0.123547  0.109278  0.111453  0.019943 -0.014937  0.044436   
2008-02-29 -0.122955 -0.062825 -0.009141 -0.037530 -0.012769 -0.015465   
2008-03-31  0.004115  0.000000  0.060386  0.091383  0.073565  0.037449   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2008-01-31  0.001965  0.023235  0.023640 -0.018693  0.012393 -0.007064   
2008-02-29  0.002768  0.042210  0.042755  0.009461  0.013373 -0.005971   
2008-03-31  0.005463 -0.008205 -0.042797  0.033969 -0.024174  0.012447   

Ticker           SPY       TLT       USO       VGK  
Date                                                
2008-01-31 -0.025842 -0.004553  0.111541 -0.002937  
2008-02-29 -0.008942  0.021388  0.011689  0.013107  
2008-03-31  0.047662 -0.024863  0.136922  0.04


### 1.6. Découpage temporel Train / Validation

Les signaux et les targets sont séparés chronologiquement conformément au protocole expérimental.

- **Train :** rendements réalisés de janvier 2010 à décembre 2017.
- **Validation :** rendements réalisés de janvier 2018 à décembre 2021.
- **Test :** rendements réalisés de janvier 2022 à décembre 2025, strictement exclus de cette analyse.

Chaque signal observé à la fin du mois `t` prédit le rendement réalisé pendant le mois `t+1`.

Les périodes de sélection sont donc décalées d'un mois par rapport aux périodes de réalisation des rendements, afin de garantir un alignement temporel correct.


In [247]:
train_signals = signals_12m.loc["2009-12-31":"2017-11-30"]
train_targets = targets_1m.loc["2009-12-31":"2017-11-30"]

print(train_signals.shape)
print(train_signals.notna().sum().sum())
print(train_targets.shape)
print(train_targets.notna().sum().sum())

print("-" * 150)

validation_signals = signals_12m.loc["2017-12-31":"2021-11-30"]
validation_targets = targets_1m.loc["2017-12-31":"2021-11-30"]

print(validation_signals.shape)
print(validation_signals.notna().sum().sum())
print(validation_targets.shape)
print(validation_targets.notna().sum().sum())

(96, 16)
1536
(96, 16)
1536
------------------------------------------------------------------------------------------------------------------------------------------------------
(48, 16)
768
(48, 16)
768



### 1.7. Nettoyage et validation des observations

Les signaux et les targets sont nettoyés à l'aide de la fonction réutilisable `clean_signals_and_targets()`.

Pour l'évaluation de classification binaire, une observation est considérée comme exploitable uniquement lorsque le signal et la target sont non manquants et différents de zéro.

Un masque booléen commun permet d'exclure les observations invalides sans modifier les dimensions ni l'alignement temporel des DataFrames.

Les nombres d'observations exploitables sont ensuite vérifiés séparément sur Train et Validation.


In [248]:
clean_train_signals, clean_train_targets, train_mask = clean_signals_and_targets(train_signals, train_targets)

clean_validation_signals, clean_validation_targets, validation_mask = clean_signals_and_targets(validation_signals, validation_targets)

print(clean_train_signals.shape)
print(clean_train_signals.notna().sum().sum())
print(clean_train_signals.isna().sum().sum())

print("-" * 10)

print(clean_train_targets.shape)
print(clean_train_targets.notna().sum().sum())
print(clean_train_targets.isna().sum().sum())

print("-" * 50)

print(clean_validation_signals.shape)
print(clean_validation_signals.notna().sum().sum())
print(clean_validation_signals.isna().sum().sum())

print("-" * 10)

print(clean_validation_targets.shape)
print(clean_validation_targets.notna().sum().sum())
print(clean_validation_targets.isna().sum().sum())

print("-" * 50) 

print(clean_train_signals.notna().equals(clean_train_targets.notna()))
print(clean_validation_signals.notna().equals(clean_validation_targets.notna()))

(96, 16)
1531
5
----------
(96, 16)
1531
5
--------------------------------------------------
(48, 16)
765
3
----------
(48, 16)
765
3
--------------------------------------------------
True
True



### Conclusion — Préparation des données

Les données nécessaires à l'analyse prédictive ont été reconstruites et validées.

- **Univers :** 16 ETF répartis entre quatre classes d'actifs.
- **Fréquence :** mensuelle, avec 168 observations de janvier 2008 à décembre 2021.
- **Signal principal :** direction du momentum historique à 12 mois.
- **Target :** direction du rendement du mois suivant.
- **Train :** 1 531 observations exploitables sur 1 536.
- **Validation :** 765 observations exploitables sur 768.

Les signaux et les targets présentent un alignement temporel cohérent et des masques de validité identiques.

Les observations neutres ou manquantes ont été exclues de l'évaluation binaire, sans modifier la structure des DataFrames.

L'échantillon Test reste strictement exclu.

**Conclusion :** les datasets Train et Validation sont prêts pour l'évaluation du pouvoir prédictif du momentum à 12 mois.



## 2. Évaluation du pouvoir prédictif

### Objectif

Évaluer la capacité du signal de Time-Series Momentum à 12 mois à prédire la direction des rendements du mois suivant.

L'analyse repose sur des métriques de classification binaire calculées séparément sur les périodes Train et Validation.

### 2.1. Accuracy

L'Accuracy mesure la proportion de prédictions directionnelles correctes parmi les observations exploitables.

Une prédiction est considérée comme correcte lorsque le signal et la target possèdent la même direction.

Les observations neutres et manquantes sont exclues du calcul à l'aide des masques de validité construits précédemment.


In [249]:
train_correct_predictions = clean_train_signals == clean_train_targets
print(train_correct_predictions.sum())
print(train_correct_predictions.sum().sum())

train_accuracy = (train_correct_predictions.sum().sum()) / (train_mask.sum().sum())
print(f"Train accuracy : {train_accuracy:.2%}")

print("-" * 150)

validation_correct_predictions = clean_validation_signals == clean_validation_targets
print(validation_correct_predictions.sum())
print(validation_correct_predictions.sum().sum())

validation_accuracy = (validation_correct_predictions.sum().sum()) / (validation_mask.sum().sum())
print(f"Validation accuracy : {validation_accuracy:.2%}")

Ticker
DBA    46
DBB    45
DBC    50
EEM    49
EWJ    50
FXA    44
FXB    38
FXE    45
FXY    57
HYG    65
IEF    45
LQD    56
SPY    65
TLT    40
USO    45
VGK    47
dtype: int64
787
Train accuracy : 51.40%
------------------------------------------------------------------------------------------------------------------------------------------------------
Ticker
DBA    30
DBB    22
DBC    26
EEM    23
EWJ    20
FXA    20
FXB    20
FXE    29
FXY    24
HYG    26
IEF    25
LQD    28
SPY    31
TLT    21
USO    25
VGK    21
dtype: int64
391
Validation accuracy : 51.11%



**Résultats — Accuracy**

Le signal de Time-Series Momentum à 12 mois obtient une Accuracy de 51,40 % sur Train (787/1 531) et de 51,11 % sur Validation (391/765).

L'écart de 0,29 point de pourcentage suggère une stabilité descriptive de cette métrique entre les deux périodes.

Toutefois, une Accuracy légèrement supérieure à 50 % ne constitue pas une preuve de pouvoir prédictif. Une comparaison avec les distributions des classes et des baselines est nécessaire.



### 2.2. Balanced Accuracy

L'Accuracy classique peut être trompeuse lorsque les deux classes (+1 et -1) sont déséquilibrées.

La **Balanced Accuracy** permet de corriger ce problème en calculant la moyenne des taux de prédiction correcte pour chaque classe (*Recall*).

Nous commencerons par analyser la répartition des directions haussières et baissières, puis nous calculerons la Balanced Accuracy sur Train et Validation.

**Objectif :** déterminer si le signal prédit correctement les deux directions du marché, plutôt que de bénéficier principalement de la fréquence d'une seule classe.


In [250]:
positive_train_targets = (clean_train_targets == 1).sum().sum()
negative_train_targets = (clean_train_targets == -1).sum().sum()

print(positive_train_targets)
print(negative_train_targets)
print((positive_train_targets + negative_train_targets) == clean_train_targets.notna().sum().sum())

print("-" * 50)

positive_correct_train_predictions = (clean_train_signals == 1) & (clean_train_targets == 1)
print(positive_correct_train_predictions.sum().sum())

negative_correct_train_predictions = (clean_train_signals == -1) & (clean_train_targets == -1)
print(negative_correct_train_predictions.sum().sum())

positive_train_recall = (positive_correct_train_predictions.sum().sum() / positive_train_targets)
negative_train_recall = (negative_correct_train_predictions.sum().sum() / negative_train_targets)

print(f"Positive train recall : {positive_train_recall:.2%}")
print(f"Negative train recall : {negative_train_recall:.2%}")

print("-" * 50)

train_balanced_accuracy = (positive_train_recall + negative_train_recall) / 2

print(f" Train balanced accuracy : {train_balanced_accuracy:.2%}")

print("-" * 150)

positive_validation_targets = (clean_validation_targets == 1).sum().sum()
negative_validation_targets = (clean_validation_targets == -1).sum().sum()

print(positive_validation_targets)
print(negative_validation_targets)
print((positive_validation_targets + negative_validation_targets) == clean_validation_targets.notna().sum().sum())

print("-" * 50)

positive_correct_validation_predictions = (clean_validation_signals == 1) & (clean_validation_targets == 1)
print(positive_correct_validation_predictions.sum().sum())

negative_correct_validation_predictions = (clean_validation_signals == -1) & (clean_validation_targets == -1)
print(negative_correct_validation_predictions.sum().sum())

positive_validation_recall = (positive_correct_validation_predictions.sum().sum() / positive_validation_targets)
negative_validation_recall = (negative_correct_validation_predictions.sum().sum() / negative_validation_targets)

print(f"Positive validation recall : {positive_validation_recall:.2%}")
print(f"Negative validation recall : {negative_validation_recall:.2%}")

print("-" * 50)

validation_balanced_accuracy = (positive_validation_recall + negative_validation_recall) / 2

print(f"Validation balanced accuracy : {validation_balanced_accuracy:.2%}")

824
707
True
--------------------------------------------------
494
293
Positive train recall : 59.95%
Negative train recall : 41.44%
--------------------------------------------------
 Train balanced accuracy : 50.70%
------------------------------------------------------------------------------------------------------------------------------------------------------
420
345
True
--------------------------------------------------
249
142
Positive validation recall : 59.29%
Negative validation recall : 41.16%
--------------------------------------------------
Validation balanced accuracy : 50.22%



**Résultats — Balanced Accuracy**

Le signal de Time-Series Momentum à 12 mois obtient :

- **Train :** Balanced Accuracy de 50,70 %.
- **Validation :** Balanced Accuracy de 50,22 %.

Les Recall révèlent une asymétrie persistante : environ 59 % des hausses sont correctement identifiées, contre seulement 41 % des baisses.

Malgré une relative stabilité entre Train et Validation, la Balanced Accuracy reste très proche de 50 %. Ces résultats ne permettent donc pas de conclure à l'existence d'un pouvoir prédictif significatif.

Une analyse des erreurs de classification et une comparaison avec des baselines sont nécessaires.


### 2.3. Confusion Matrix

La **Confusion Matrix** permet d'analyser les erreurs de classification du signal de Time-Series Momentum à 12 mois.

Elle distingue quatre situations :

- **True Positive (TP)** : hausse correctement prédite.
- **True Negative (TN)** : baisse correctement prédite.
- **False Positive (FP)** : hausse prédite, mais baisse réalisée.
- **False Negative (FN)** : baisse prédite, mais hausse réalisée.

Nous construirons les matrices de confusion sur Train et Validation afin d'identifier les types d'erreurs commises par le signal et d'évaluer leur stabilité entre les deux périodes.

**Objectif :** compléter l'Accuracy et la Balanced Accuracy par une analyse détaillée des prédictions correctes et incorrectes.


In [251]:
train_true_positive = positive_correct_train_predictions
train_true_negative = negative_correct_train_predictions
train_false_positive = (clean_train_signals == 1) & (clean_train_targets == -1)
train_false_negative = (clean_train_signals == -1) & (clean_train_targets == 1)

print(f"Train true positive predictions : {train_true_positive.sum().sum()}")
print(f"Train false positive predictions : {train_false_positive.sum().sum()}")
print(f"Sum of train positive predictions : {(train_true_positive.sum().sum()) + (train_false_positive.sum().sum())}")

print ("")

print(f"Train true negative predictions : {train_true_negative.sum().sum()}")
print(f"Train false negative predictions : {train_false_negative.sum().sum()}")
print(f"Sum of train negative predictions : {(train_true_negative.sum().sum()) + (train_false_negative.sum().sum())}")

print(((train_true_positive.sum().sum()) + (train_false_positive.sum().sum()) + (train_true_negative.sum().sum()) + (train_false_negative.sum().sum())) == clean_train_targets.notna().sum().sum())

print("")
print("-" * 50)
print("")

validation_true_positive = positive_correct_validation_predictions
validation_true_negative = negative_correct_validation_predictions
validation_false_positive = (clean_validation_signals == 1) & (clean_validation_targets == -1)
validation_false_negative = (clean_validation_signals == -1) & (clean_validation_targets == 1)

print(f"Validation true positive predictions : {validation_true_positive.sum().sum()}")
print(f"Validation false positive predictions : {validation_false_positive.sum().sum()}")
print(f"Sum of validation positive predictions : {(validation_true_positive.sum().sum()) + (validation_false_positive.sum().sum())}")

print("")

print(f"Validation true negative predictions : {validation_true_negative.sum().sum()}")
print(f"Validation false negative predictions : {validation_false_negative.sum().sum()}")
print(f"Sum of validation negative predictions : {(validation_true_negative.sum().sum()) + (validation_false_negative.sum().sum())}")

print(((validation_true_positive.sum().sum()) + (validation_false_positive.sum().sum()) + (validation_true_negative.sum().sum()) + (validation_false_negative.sum().sum())) == clean_validation_targets.notna().sum().sum())

Train true positive predictions : 494
Train false positive predictions : 414
Sum of train positive predictions : 908

Train true negative predictions : 293
Train false negative predictions : 330
Sum of train negative predictions : 623
True

--------------------------------------------------

Validation true positive predictions : 249
Validation false positive predictions : 203
Sum of validation positive predictions : 452

Validation true negative predictions : 142
Validation false negative predictions : 171
Sum of validation negative predictions : 313
True



**Matrices de confusion — Train / Validation**

Les résultats sont maintenant organisés sous forme de matrices de confusion 2 × 2.

- **Lignes :** directions prédites par le signal.
- **Colonnes :** directions réellement observées.
- **Diagonale principale (TP, TN) :** prédictions correctes.
- **Autres cellules (FP, FN) :** prédictions incorrectes.

Cette représentation permet d'identifier les erreurs directionnelles et de vérifier la cohérence des métriques de classification calculées précédemment.


In [252]:
import pandas as pd

train_confusion_matrix = pd.DataFrame(
                            data=[[train_true_positive.sum().sum(), train_false_positive.sum().sum()],
                                [train_false_negative.sum().sum(), train_true_negative.sum().sum()]
                                ],
                                index=["Signal +1", "Signal -1"],
                                columns=["Target +1", "Target -1"]    
                        )

validation_confusion_matrix = pd.DataFrame(
                                data=[[validation_true_positive.sum().sum(), validation_false_positive.sum().sum()],
                                    [validation_false_negative.sum().sum(), validation_true_negative.sum().sum()]
                                    ],
                                    index=["Signal +1", "Signal -1"],
                                    columns=["Target +1", "Target -1"]
                            )

print(train_confusion_matrix)
print("")
print("")
print(validation_confusion_matrix)

print("")
print("")

print(train_confusion_matrix.sum(axis=0))
print(validation_confusion_matrix.sum(axis=0))

           Target +1  Target -1
Signal +1        494        414
Signal -1        330        293


           Target +1  Target -1
Signal +1        249        203
Signal -1        171        142


Target +1    824
Target -1    707
dtype: int64
Target +1    420
Target -1    345
dtype: int64



**Synthèse des classifications — Train / Validation**

Le tableau suivant regroupe les quatre catégories de prédictions (TP, FP, TN, FN) pour les périodes Train et Validation.

Cette présentation facilite la comparaison des erreurs et des prédictions correctes entre les deux échantillons.

Les résultats sont calculés directement à partir des observations valides, sans saisir les valeurs manuellement.


In [253]:
confusion_summary = {"Train" :
                          {"True positive predictions" : train_true_positive.sum().sum(),
                           "False positive predictions" : train_false_positive.sum().sum(),
                           "True negative predictions" : train_true_negative.sum().sum(),
                           "False negative predictions" : train_false_negative.sum().sum()},
                    "Validation" :
                           {"True positive predictions" : validation_true_positive.sum().sum(),
                            "False positive predictions" : validation_false_positive.sum().sum(),
                            "True negative predictions" : validation_true_negative.sum().sum(),
                            "False negative predictions" : validation_false_negative.sum().sum()}
                          }

confusion_summary_df = pd.DataFrame(confusion_summary).T
confusion_summary_df

,True positive predictions,False positive predictions,True negative predictions,False negative predictions
Train,494,414,293,330
Validation,249,203,142,171



**Résultats et interprétation — Confusion Matrix**

Sur Train, le signal réalise 787 prédictions correctes contre 744 incorrectes. Sur Validation, il réalise 391 prédictions correctes contre 374 incorrectes.

Le signal présente une tendance à prédire davantage de hausses que de baisses :

- **Train :** 908 prédictions haussières contre 623 baissières.
- **Validation :** 452 prédictions haussières contre 313 baissières.

Cette asymétrie est cohérente avec les Recall observés précédemment : le signal identifie mieux les hausses que les baisses.

La structure des erreurs reste relativement similaire entre Train et Validation. Cependant, ces résultats ne démontrent pas à eux seuls l'existence d'un avantage prédictif.

**Conclusion :** l'analyse des matrices de confusion confirme une capacité de classification limitée et un biais directionnel haussier. Une comparaison avec des baselines est nécessaire pour évaluer la valeur ajoutée du signal.



### 2.4. Baseline Comparisons

L'évaluation du pouvoir prédictif nécessite de comparer le signal de Time-Series Momentum à des modèles de référence simples (*baselines*).

Une Accuracy supérieure à 50 % ne constitue pas nécessairement un avantage prédictif, notamment lorsque les rendements haussiers et baissiers ne sont pas également représentés.

Nous comparerons le signal à des prédictions directionnelles naïves, sur les mêmes observations exploitables, séparément pour Train et Validation.

**Objectif :** déterminer si le momentum à 12 mois apporte une valeur prédictive supplémentaire par rapport à des modèles ne reposant sur aucune information de momentum.


In [254]:
train_always_long_accuracy = (positive_train_targets / train_mask.sum().sum())
print(f"Train always long accuracy : {train_always_long_accuracy:.2%}")

train_always_short_accuracy = (negative_train_targets / train_mask.sum().sum())
print(f"Train always negative accuracy : {train_always_short_accuracy:.2%}")

train_always_long_balanced_accuracy = (1 + 0) / 2 
print(f"Train always long balanced accuracy : {train_always_long_balanced_accuracy:.2%}")

print("-" * 50)

validation_always_long_accuracy = (positive_validation_targets / validation_mask.sum().sum())
print(f"Validation always long accuracy : {validation_always_long_accuracy:.2%}")

validation_always_short_accuracy = (negative_validation_targets / validation_mask.sum().sum())
print(f"Validation always short accuracy : {validation_always_short_accuracy:.2%}")

validation_always_long_balanced_accuracy = (1 + 0) / 2
print(f"Validation always long balanced accuracy : {validation_always_long_balanced_accuracy:.2%}")

Train always long accuracy : 53.82%
Train always negative accuracy : 46.18%
Train always long balanced accuracy : 50.00%
--------------------------------------------------
Validation always long accuracy : 54.90%
Validation always short accuracy : 45.10%
Validation always long balanced accuracy : 50.00%



**Résultats et interprétation — Baseline Comparisons**

Le signal de Time-Series Momentum à 12 mois est comparé à deux baselines directionnelles : Always Long (+1) et Always Short (-1).

**Résultats :**
- **Train :** le momentum obtient 51,40 % d'Accuracy, contre 53,82 % pour Always Long.
- **Validation :** le momentum obtient 51,11 % d'Accuracy, contre 54,90 % pour Always Long.
- Les deux baselines possèdent une Balanced Accuracy de 50 %.
- Le momentum atteint une Balanced Accuracy de 50,70 % sur Train et de 50,22 % sur Validation.

Le momentum ne surpasse donc pas la baseline Always Long en Accuracy. Son léger avantage en Balanced Accuracy reste insuffisant pour établir un pouvoir prédictif statistiquement significatif.

**Conclusion :** les résultats descriptifs ne démontrent pas de valeur prédictive robuste du signal à 12 mois. Une analyse de stabilité et des tests statistiques adaptés seront nécessaires avant toute conclusion définitive.



## 3. Train / Validation Comparison

### Objectif

Comparer les performances prédictives du signal de Time-Series Momentum à 12 mois entre les périodes Train (2010–2017) et Validation (2018–2021).

L'analyse vise à :

- Consolider les principales métriques de classification dans un tableau comparatif.
- Mesurer les écarts de performance entre Train et Validation.
- Examiner la stabilité des Recall haussier et baissier.
- Comparer les performances du momentum aux baselines directionnelles.
- Identifier les limites méthodologiques des résultats.

**Attention :** une stabilité descriptive entre les deux périodes ne démontre pas, à elle seule, l'existence d'un pouvoir prédictif statistiquement significatif.


In [276]:
performance_summary = {"Train" :
                       {"Accuracy" : train_accuracy,
                        "Balanced Accuracy" : train_balanced_accuracy,
                        "Positive Recall" : positive_train_recall,
                        "Negative Recall" : negative_train_recall,
                        "Always Long Accuracy" : train_always_long_accuracy,
                        "Always Short Accuracy" : train_always_short_accuracy},
                        "Validation":
                        {"Accuracy" : validation_accuracy,
                         "Balanced Accuracy" : validation_balanced_accuracy,
                         "Positive Recall" : positive_validation_recall, 
                         "Negative Recall" : negative_validation_recall,
                         "Always Long Accuracy" : validation_always_long_accuracy,
                         "Always Short Accuracy" : validation_always_short_accuracy}
                        }

performance_summary_df = pd.DataFrame(performance_summary)
# display(performance_summary_df.style.format("{:.2%}"))

performance_summary_df["Difference (pp)"] = (performance_summary_df["Validation"] - performance_summary_df["Train"]) * 100
display(performance_summary_df.style.format({"Train" : "{:.2%}",
                                             "Validation" : "{:.2%}",
                                             "Difference (pp)" : "{:+.4f}"
                                             }))

,Train,Validation,Difference (pp)
Accuracy,51.40%,51.11%,-0.2932
Balanced Accuracy,50.70%,50.22%,-0.4745
Positive Recall,59.95%,59.29%,-0.6657
Negative Recall,41.44%,41.16%,-0.2833
Always Long Accuracy,53.82%,54.90%,+1.0809
Always Short Accuracy,46.18%,45.10%,-1.0809



### Conclusion — Train / Validation Comparison

La comparaison des performances prédictives du signal de Time-Series Momentum à 12 mois révèle une stabilité descriptive entre Train (2010–2017) et Validation (2018–2021).

**Principaux résultats :**

- L'Accuracy passe de 51,40 % à 51,11 %, soit une différence de -0,29 point de pourcentage.
- La Balanced Accuracy passe de 50,70 % à 50,22 %, soit une différence d'environ -0,47 point.
- Les Recall haussier et baissier restent relativement stables, avec une meilleure détection des hausses que des baisses.
- La baseline Always Long surpasse le momentum en Accuracy sur les deux périodes, notamment en raison de la fréquence supérieure des rendements positifs.

**Interprétation :**

Le signal présente un comportement directionnel relativement stable entre les deux échantillons. Cependant, cette stabilité concerne des performances prédictives faibles, proches du niveau de référence de 50 % en Balanced Accuracy.

Les résultats ne fournissent pas, à ce stade, de preuve convaincante d'un avantage prédictif du momentum à 12 mois.

Cette analyse reste descriptive. Elle ne tient pas encore compte de l'incertitude statistique, des dépendances temporelles et cross-asset, ni des performances économiques d'une stratégie.

**Conclusion :** la robustesse du signal ne peut pas être établie sur la seule base de cette comparaison. Des analyses complémentaires par classe d'actifs, lookback et période seront nécessaires.
